In [0]:
df = spark.read.table("gizmobox.bronze.py_customers")
display(df)

In [0]:
df_filtered = spark.table("gizmobox.bronze.py_customers").filter("customer_id is not null")
display(df_filtered)

In [0]:
df_distinct = df_filtered.dropDuplicates()
display(df_distinct)


In [0]:
from pyspark.sql.functions import max  
df_max_timestamp =   (
    df.distinct().groupBy("customer_id")
    .agg(max("created_timestamp").alias("max_timestamp"))
)

In [0]:
df_distinct_customer = (
    df_distinct
      .join(df_max_timestamp,
            (df_distinct.customer_id == df_max_timestamp.customer_id) &
            (df_distinct.created_timestamp == df_max_timestamp.max_timestamp),
            "inner")
      .select(df_distinct["*"])
)
display(df_distinct_customer)

In [0]:
df_casted_customer = (
    df_distinct_customer
    .select(df_distinct_customer.created_timestamp.cast("timestamp"),
            df_distinct_customer.customer_id.cast("int"),
            df_distinct_customer.email.cast("string"),
            df_distinct_customer.customer_name.cast("string"),
            df_distinct_customer.date_of_birth.cast("date"),
            df_distinct_customer.telephone,
            df_distinct_customer.member_since.cast("date")
         )
)
display(df_casted_customer)

In [0]:
df_casted_customer.write.mode("overwrite").saveAsTable("gizmobox.silver.py_customers")

In [0]:
%sql
select * from gizmobox.silver.py_customers